In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
import sys

In [2]:
file_path = "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx"
attendance_df = pd.read_excel(file_path,sheet_name="attendance_monthly_features")

In [3]:
attendance_df.shape

(7339, 16)

In [4]:
attendance_df.columns.tolist()

['attendance_feature_id',
 'employee_id',
 'month',
 'department_id',
 'job_level',
 'scheduled_days',
 'present_days',
 'wfh_days',
 'half_days_unpaid',
 'paid_leave_days',
 'unpaid_absence_days',
 'longest_unpaid_streak_days',
 'late_arrivals',
 'avg_hours_worked',
 'overtime_hours',
 'attendance_rate']

In [5]:
attendance_df.dtypes

attendance_feature_id                    str
employee_id                              str
month                         datetime64[us]
department_id                            str
job_level                              int64
scheduled_days                         int64
present_days                           int64
wfh_days                               int64
half_days_unpaid                       int64
paid_leave_days                        int64
unpaid_absence_days                    int64
longest_unpaid_streak_days             int64
late_arrivals                          int64
avg_hours_worked                     float64
overtime_hours                       float64
attendance_rate                      float64
dtype: object

In [6]:
attendance_df.head

<bound method NDFrame.head of      attendance_feature_id employee_id      month department_id  job_level  \
0                 AF-00001    EMP-0002 2025-04-01           D08          4   
1                 AF-00002    EMP-0002 2025-05-01           D08          4   
2                 AF-00003    EMP-0002 2025-06-01           D08          4   
3                 AF-00004    EMP-0002 2025-07-01           D08          4   
4                 AF-00005    EMP-0002 2025-08-01           D08          4   
...                    ...         ...        ...           ...        ...   
7334              AF-07335    EMP-0741 2026-09-01           D01          3   
7335              AF-07336    EMP-0742 2026-09-01           D04          3   
7336              AF-07337    EMP-0743 2026-09-01           D09          2   
7337              AF-07338    EMP-0744 2026-09-01           D01          2   
7338              AF-07339    EMP-0745 2026-09-01           D06          1   

      scheduled_days  present_day

In [15]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score


numeric_columns = attendance_df.select_dtypes(include="number").columns.tolist()

exclude_columns = [
    col for col in numeric_columns
    if col.lower().endswith("_id")
    or col.lower() in ["id", "row_id", "month", "date", "observation_date"]
]

attendance_features = attendance_df[numeric_columns].drop(
    columns=exclude_columns,
    errors="ignore"
)

attendance_features = attendance_features.fillna(
    attendance_features.median()
)

print("Features used:", attendance_features.columns.tolist())



Features used: ['job_level', 'scheduled_days', 'present_days', 'wfh_days', 'half_days_unpaid', 'paid_leave_days', 'unpaid_absence_days', 'longest_unpaid_streak_days', 'late_arrivals', 'avg_hours_worked', 'overtime_hours', 'attendance_rate', 'anomaly_flag', 'anomaly_score', 'combined_anomaly_flag']


In [16]:
model = IsolationForest(
    contamination=0.03,
    random_state=42
)
predictions = model.fit_predict(attendance_features)
attendance_df["anomaly_flag"] = (predictions == -1).astype(int)

attendance_df["anomaly_score"] = -model.decision_function(
    attendance_features
)

print("\nTotal records:", len(attendance_df))
print("Anomalies flagged:", attendance_df["anomaly_flag"].sum())
print("Anomaly percentage:",
      round(attendance_df["anomaly_flag"].mean() * 100, 2), "%")


Total records: 7339
Anomalies flagged: 221
Anomaly percentage: 3.01 %


In [10]:
# attendance rule

In [11]:
attendance_df["zero_attendance"] = (
    attendance_df["attendance_rate"] == 0
)
attendance_df["invalid_attendance"] = (
    attendance_df["attendance_rate"].notna()
    & ~attendance_df["attendance_rate"].between(0, 1)
)
attendance_df["rule_flag"] = (
    attendance_df["zero_attendance"]
    | attendance_df["invalid_attendance"]
)

attendance_df["combined_anomaly_flag"] = (
    (attendance_df["anomaly_flag"] == 1)
    | attendance_df["rule_flag"]
).astype(int)
print("Zero attendance:", attendance_df["zero_attendance"].sum())
print("Invalid attendance rate:", attendance_df["invalid_attendance"].sum())
print("Rule-based flags:", attendance_df["rule_flag"].sum())
print("Isolation Forest flags:", attendance_df["anomaly_flag"].sum())
print("Combined flags:", attendance_df["combined_anomaly_flag"].sum())

Zero attendance: 50
Invalid attendance rate: 0
Rule-based flags: 50
Isolation Forest flags: 221
Combined flags: 221


In [13]:
file_path = "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx"
labels_df = pd.read_excel(
    file_path,
    sheet_name="attendance_anomaly_labels"
)
print("Shape:", labels_df.shape)
print("Columns:", labels_df.columns.tolist())
labels_df.head()

Shape: (150, 5)
Columns: ['attendance_feature_id', 'employee_id', 'month', 'anomaly_type', 'injection_note']


,attendance_feature_id,employee_id,month,anomaly_type,injection_note
0,AF-04604,EMP-0484,2025-12-01,ABSENCE_STREAK,10 consecutive unpaid absence days
1,AF-02143,EMP-0249,2026-07-01,ABSENCE_STREAK,13 consecutive unpaid absence days
2,AF-01814,EMP-0208,2026-01-01,ABSENCE_STREAK,9 consecutive unpaid absence days
3,AF-00927,EMP-0100,2025-10-01,ABSENCE_STREAK,14 consecutive unpaid absence days
4,AF-03762,EMP-0414,2025-12-01,ABSENCE_STREAK,12 consecutive unpaid absence days


In [14]:
print("Attendance feature ID available:",
      "attendance_feature_id" in attendance_df.columns)

print("Unique labeled anomalies:",
      labels_df["attendance_feature_id"].nunique())

print("Matched label records:",
      labels_df["attendance_feature_id"].isin(
          attendance_df["attendance_feature_id"]
      ).sum())

print("Duplicate attendance IDs:",
      attendance_df["attendance_feature_id"].duplicated().sum())

Attendance feature ID available: True
Unique labeled anomalies: 150
Matched label records: 150
Duplicate attendance IDs: 0


In [18]:
attendance_df["true_anomaly"] = ( attendance_df["attendance_feature_id"].isin( labels_df["attendance_feature_id"]).astype(int))

In [19]:
y_true = attendance_df["true_anomaly"]
y_pred = attendance_df["combined_anomaly_flag"]

In [20]:
def catch_rate(y_true, y_pred):
    detected = ((y_true == 1) & (y_pred == 1)).sum()
    total = (y_true == 1).sum()
    return detected / total

In [21]:
 y_true.sum()

np.int64(150)

In [23]:
(y_true == 1) & (y_pred == 1).sum()

0       False
1       False
2       False
3       False
4       False
        ...  
7334    False
7335    False
7336    False
7337    False
7338    False
Name: true_anomaly, Length: 7339, dtype: bool

In [24]:
round(catch_rate(y_true, y_pred), 3)

np.float64(0.58)

In [25]:
 round(precision_score(y_true, y_pred, zero_division=0), 3)

0.394

In [26]:
print("Known anomalies:", y_true.sum())
print("Anomalies detected:", ((y_true == 1) & (y_pred == 1)).sum())
print("Catch rate / Recall:", round(catch_rate(y_true, y_pred), 3))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 3))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 3))

Known anomalies: 150
Anomalies detected: 87
Catch rate / Recall: 0.58
Precision: 0.394
Recall: 0.58
F1-score: 0.469


In [29]:
evaluation_df = labels_df.merge(attendance_df[["attendance_feature_id", "combined_anomaly_flag"]],on="attendance_feature_id", how="left")
missed = evaluation_df[evaluation_df["combined_anomaly_flag"] == 0]
print("Missed anomalies:", len(missed))
missed["anomaly_type"].value_counts().to_frame("count")
print("\nExamples of missed anomalies:")
missed[["attendance_feature_id", "anomaly_type", "injection_note" ]].head(10)

Missed anomalies: 63

Examples of missed anomalies:


,attendance_feature_id,anomaly_type,injection_note
55,AF-02162,IMPOSSIBLE_HOURS,average 18.88 hours per day
57,AF-03659,IMPOSSIBLE_HOURS,average 15.14 hours per day
58,AF-02469,IMPOSSIBLE_HOURS,average 14.7 hours per day
59,AF-03394,IMPOSSIBLE_HOURS,average 18.24 hours per day
60,AF-04096,IMPOSSIBLE_HOURS,average 18.4 hours per day
61,AF-00148,IMPOSSIBLE_HOURS,average 15.18 hours per day
62,AF-06658,IMPOSSIBLE_HOURS,average 16.02 hours per day
63,AF-01427,IMPOSSIBLE_HOURS,average 14.62 hours per day
64,AF-02060,IMPOSSIBLE_HOURS,average 17.35 hours per day
65,AF-02881,IMPOSSIBLE_HOURS,average 17.12 hours per day


In [ ]:
# Task 5 — Attendance Anomaly Detection Report
#I used Isolation Forest with 3% contamination to detect unusual attendance records without training on anomaly labels. I then combined the model's predictions with simple rules for zero attendance and invalid attendance rates. The combined detector flagged 221 records out of 7,339.

#When evaluated against the 150 known labeled anomalies, the detector identified 87 and missed 63. It achieved a precision of 0.394, a recall of 0.580, and an F1-score of 0.469, which are close to the supplied precision and recall benchmarks of approximately 0.40 and 0.59.

#The missed anomalies consisted of 34 IMPOSSIBLE_HOURS cases and 29 LATE_SPIKE cases. Impossible working hours may require explicit validation rules, while late-spike detection may benefit from comparing an employee's attendance against their historical pattern. These are possible improvements that should be tested separately.

#The model and rules were developed without using the labels for training. The labels were used only to evaluate the detector. Since the label sheet lists known anomalies, precision should be interpreted according to the assignment's labeling setup and may not represent the detector's true precision if other records contain unlisted anomalies.